# `generate_code` — Exhaustive Path Demo

This notebook exercises **every code path** in `generate_code_from_plan` by
constructing `DataProfile` + `ForecastPlan` pairs and printing the generated code.

The generated code is **not executed** — only printed and syntax-checked via `compile()`.

In [1]:
import sys
sys.path.insert(0, "..")

from skforecast_ai.schemas import DataProfile, ForecastPlan, PreprocessingStep
from skforecast_ai import ForecastingAssistant

assistant = ForecastingAssistant()


def show(plan, profile):
    """Generate code, syntax-check it, and print."""
    code = assistant.generate_code_from_plan(plan=plan, data_profile=profile)
    compile(code, "<demo>", "exec")  # raises SyntaxError if invalid
    print(code)
    print("\n" + "=" * 70 + "\n")

## 1. Single Series

### 1.1 Recursive minimal (LGBMRegressor, no exog, no intervals)

In [2]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 365,
    target         = "y",
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "single_series",
    forecaster        = "ForecasterRecursive",
    estimator         = "LGBMRegressor",
    steps             = 30,
    frequency         = "D",
    forecaster_kwargs = {"lags": [1, 2, 3, 4, 5, 6, 7], "dropna_from_series": False},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Minimal recursive.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator          = LGBMRegressor(random_state=123, verbose=-1),
    lags               = [1, 2, 3, 4, 5, 6, 7],
    dropna_from_series = False,
)

# Fit
forecaster.fit(y=data_train['y'])

# Predict
steps = 30
predictions

### 1.2 Recursive full (Ridge, exog+categorical, window_features, transformers, differentiation)

In [3]:
profile = DataProfile(
    n_series         = 1,
    n_observations   = 720,
    target           = "sales",
    index_type       = "datetime",
    frequency        = "h",
    exog_columns     = ["temperature", "day_of_week"],
    categorical_exog = ["day_of_week"],
    end_train        = "2024-01-24",
)

plan = ForecastPlan(
    task_type         = "single_series",
    forecaster        = "ForecasterRecursive",
    estimator         = "Ridge",
    steps             = 24,
    frequency         = "h",
    forecaster_kwargs = {
        "lags": [1, 7, 24],
        "window_features": [
            {"stats": ["mean", "std"], "window_sizes": 24},
            {"stats": ["mean"], "window_sizes": 168},
        ],
        "transformer_y": "StandardScaler",
        "transformer_exog": "StandardScaler",
        "categorical_features": "auto",
        "differentiation": 1,
        "dropna_from_series": True,
    },
    interval          = [5, 95],
    interval_method   = "bootstrapping",
    use_exog          = True,
    explanation       = "Full config.",
)

show(plan, profile)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('h')

# Train/test split
end_train = '2024-01-24'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]
exog_features = ['temperature', 'day_of_week']

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean

### 1.3 Recursive exog numeric-only (plain StandardScaler transformer_exog)

In [4]:
profile = DataProfile(
    n_series         = 1,
    n_observations   = 720,
    target           = "sales",
    index_type       = "datetime",
    frequency        = "h",
    exog_columns     = ["temperature", "humidity"],
    categorical_exog = [],
    end_train        = "2024-01-24",
)

plan = ForecastPlan(
    task_type         = "single_series",
    forecaster        = "ForecasterRecursive",
    estimator         = "LGBMRegressor",
    steps             = 24,
    frequency         = "h",
    forecaster_kwargs = {
        "lags": [1, 2, 3, 24],
        "transformer_exog": "StandardScaler",
        "dropna_from_series": False,
    },
    interval_method   = "bootstrapping",
    use_exog          = True,
    explanation       = "Exog numeric only.",
)

show(plan, profile)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('h')

# Train/test split
end_train = '2024-01-24'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]
exog_features = ['temperature', 'humidity']

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

transformer_exog = StandardScaler()

# Create forecaster
forecaster = ForecasterRecursive(
    estimator          = LGBMRegressor(random_state=123, verbose=-1),
    lags               = [

### 1.4 ForecasterDirect (steps in constructor)

In [5]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 365,
    target         = "y",
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "single_series",
    forecaster        = "ForecasterDirect",
    estimator         = "Ridge",
    steps             = 14,
    frequency         = "D",
    forecaster_kwargs = {"lags": [1, 2, 3, 4, 5, 6, 7], "steps": 14, "dropna_from_series": False},
    interval_method   = "bootstrapping",
    use_exog          = False,
    explanation       = "Direct forecaster.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.direct import ForecasterDirect

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster
forecaster = ForecasterDirect(
    estimator          = Ridge(),
    steps              = 14,
    lags               = [1, 2, 3, 4, 5, 6, 7],
    dropna_from_series = False,
)

# Fit
forecaster.fit(
    y                         = data_train['y'],
    store_in_s

### 1.5 Unknown estimator (TODO import)

In [6]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 365,
    target         = "y",
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "single_series",
    forecaster        = "ForecasterRecursive",
    estimator         = "GradientBoostingRegressor",
    steps             = 10,
    frequency         = "D",
    forecaster_kwargs = {"lags": [1, 2, 3], "dropna_from_series": False},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Unknown estimator.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from __main__ import GradientBoostingRegressor  # TODO: replace with correct import for GradientBoostingRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator          = GradientBoostingRegressor(random_state=123),
    lags               = [1, 2, 3],
    dropna_from_series = False,
)

# Fit


### 1.6 Preprocessing steps (sort_index + asfreq)

In [7]:
profile = DataProfile(
    n_series           = 1,
    n_observations     = 365,
    target             = "y",
    index_type         = "datetime",
    frequency          = "D",
    frequency_is_set   = False,
    index_is_monotonic = False,
    end_train          = "2024-10-18",
)

plan = ForecastPlan(
    task_type           = "single_series",
    forecaster          = "ForecasterRecursive",
    estimator           = "LGBMRegressor",
    steps               = 10,
    frequency           = "D",
    forecaster_kwargs   = {"lags": [1, 2, 3, 7], "dropna_from_series": False},
    interval_method     = None,
    use_exog            = False,
    preprocessing_steps = [
        PreprocessingStep(
            action="sort_index",
            reason="Index not monotonic.",
            code_snippet="data = data.sort_index()",
            blocking=True,
        ),
        PreprocessingStep(
            action="asfreq",
            reason="Frequency not set.",
            code_snippet="data = data.asfreq('{frequency}')",
            blocking=True,
        ),
    ],
    explanation = "With preprocessing.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)

# Preprocessing
data = data.sort_index()
data = data.asfreq('D')

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator          = LGBMRegressor(random_state=123, verbose=-1),
    lags               = [1, 2, 3, 7],
    dropna_from_series = False,
)

# Fit
forecaster.fit(y=data_train['y'])


## 2. Multi-Series

### 2.1 Long format (pivot_table, conformal intervals)

In [8]:
profile = DataProfile(
    data_format      = "long",
    n_series         = 3,
    n_observations   = 300,
    target           = "sales",
    date_column      = "date",
    series_id_column = "series_id",
    index_type       = "datetime",
    frequency        = "D",
    end_train        = "2024-08-05",
)

plan = ForecastPlan(
    task_type         = "multi_series",
    forecaster        = "ForecasterRecursiveMultiSeries",
    estimator         = "LGBMRegressor",
    steps             = 14,
    frequency         = "D",
    forecaster_kwargs = {"lags": [1, 2, 3, 4, 5, 6, 7], "encoding": "ordinal", "dropna_from_series": False},
    interval_method   = "conformal",
    use_exog          = False,
    explanation       = "Long format multi-series.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.preprocessing import reshape_series_long_to_dict
from skforecast.recursive import ForecasterRecursiveMultiSeries

# Load data (long format)
data = pd.read_csv('data.csv', parse_dates=['date'])

# Reshape to dict format (optimal for ForecasterRecursiveMultiSeries)
series_dict = reshape_series_long_to_dict(
    data      = data,
    freq      = 'D',
    series_id = 'series_id',
    index     = 'date',
    values    = 'sales',
)

# Train/test split
end_train = '2024-08-05'  # 80% of data, adjust to change the split point
series_dict_train = {k: v.loc[:end_train] for k, v in series_dict.items()}
series_dict_test  = {k: v.loc[v.index > end_train] for k, v in series_dict.items()}

# Create forecaster
forecaster = ForecasterRecursiveMultiSeries(
    estimator          = LGBMRegressor(random_state=12

### 2.2 Wide format (no pivot, no intervals)

In [9]:
profile = DataProfile(
    data_format    = "wide",
    n_series       = 3,
    n_observations = 300,
    target         = ["series_a", "series_b", "series_c"],
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-08-05",
)

plan = ForecastPlan(
    task_type         = "multi_series",
    forecaster        = "ForecasterRecursiveMultiSeries",
    estimator         = "LGBMRegressor",
    steps             = 14,
    frequency         = "D",
    forecaster_kwargs = {"lags": [1, 2, 3, 7], "encoding": "ordinal", "dropna_from_series": False},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Wide format multi-series.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.recursive import ForecasterRecursiveMultiSeries

# Load data (wide format)
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

# Reshape to dict format (optimal for ForecasterRecursiveMultiSeries)
series_dict = data[['series_a', 'series_b', 'series_c']].to_dict('series')

# Train/test split
end_train = '2024-08-05'  # 80% of data, adjust to change the split point
series_dict_train = {k: v.loc[:end_train] for k, v in series_dict.items()}
series_dict_test  = {k: v.loc[v.index > end_train] for k, v in series_dict.items()}

# Create forecaster
forecaster = ForecasterRecursiveMultiSeries(
    estimator          = LGBMRegressor(random_state=123, verbose=-1),
    lags               = [1, 2, 3, 7],
    encoding           = 'ordinal',
    dropna_from_series = False,


### 2.3 Long format with exog+categorical (transformer_series, transformer_exog, window_features)

In [10]:
profile = DataProfile(
    data_format      = "long",
    n_series         = 3,
    n_observations   = 300,
    target           = "sales",
    date_column      = "date",
    series_id_column = "store_id",
    index_type       = "datetime",
    frequency        = "D",
    exog_columns     = ["temperature", "holiday"],
    categorical_exog = ["holiday"],
    end_train        = "2024-08-05",
)

plan = ForecastPlan(
    task_type         = "multi_series",
    forecaster        = "ForecasterRecursiveMultiSeries",
    estimator         = "LGBMRegressor",
    steps             = 7,
    frequency         = "D",
    forecaster_kwargs = {
        "lags": [1, 2, 3, 7],
        "encoding": "ordinal",
        "window_features": [{"stats": ["mean", "std"], "window_sizes": 7}],
        "transformer_series": "StandardScaler",
        "transformer_exog": "StandardScaler",
        "categorical_features": "auto",
        "differentiation": 1,
        "dropna_from_series": False,
    },
    interval_method   = None,
    use_exog          = True,
    explanation       = "Multi-series full config.",
)

show(plan, profile)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.preprocessing import RollingFeatures, reshape_series_long_to_dict, reshape_exog_long_to_dict
from skforecast.recursive import ForecasterRecursiveMultiSeries

# Load data (long format)
data = pd.read_csv('data.csv', parse_dates=['date'])

# Reshape to dict format (optimal for ForecasterRecursiveMultiSeries)
series_dict = reshape_series_long_to_dict(
    data      = data,
    freq      = 'D',
    series_id = 'store_id',
    index     = 'date',
    values    = 'sales',
)

exog_dict = reshape_exog_long_to_dict(
    data      = data[['store_id', 'date', 'temperature', 'holiday']],
    freq      = 'D',
    series_id = 'store_id',
    index     = 'date',
)

# Train/test split
end_train = '2024-08-05'

## 3. Multivariate

### 3.1 Wide, no exog

In [11]:
profile = DataProfile(
    data_format    = "wide",
    n_series       = 3,
    n_observations = 365,
    target         = ["series_a", "series_b", "series_c"],
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "multivariate",
    forecaster        = "ForecasterDirectMultiVariate",
    estimator         = "Ridge",
    steps             = 10,
    frequency         = "D",
    forecaster_kwargs = {"lags": [1, 2, 3], "steps": 10, "dropna_from_series": False},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Multivariate basic.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.direct import ForecasterDirectMultiVariate

# Load data (wide format)
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

series = data[['series_a', 'series_b', 'series_c']]

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
series_train = series.loc[:end_train]
series_test  = series.loc[series.index > end_train]

# Create forecaster
forecaster = ForecasterDirectMultiVariate(
    estimator          = Ridge(),
    level              = 'series_a',
    steps              = 10,
    lags               = [1, 2, 3],
    dropna_from_series = False,
)

# Fit
forecaster.fit(series=series_train)

# Predict
steps = 10
predictions = forecaster.predict(steps=steps)
print(predictions)

# Evaluate on test set
actual = series_te

### 3.2 Wide, with exog, intervals, differentiation

In [12]:
profile = DataProfile(
    data_format    = "wide",
    n_series       = 3,
    n_observations = 365,
    target         = ["series_a", "series_b", "series_c"],
    index_type     = "datetime",
    frequency      = "D",
    exog_columns   = ["temperature", "promo"],
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "multivariate",
    forecaster        = "ForecasterDirectMultiVariate",
    estimator         = "LGBMRegressor",
    steps             = 10,
    frequency         = "D",
    forecaster_kwargs = {
        "lags": [1, 2, 3],
        "steps": 10,
        "differentiation": 1,
        "dropna_from_series": False,
    },
    interval_method   = "bootstrapping",
    use_exog          = True,
    explanation       = "Multivariate with exog and intervals.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.direct import ForecasterDirectMultiVariate

# Load data (wide format)
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

series = data[['series_a', 'series_b', 'series_c']]

exog = data[['temperature', 'promo']]

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
series_train = series.loc[:end_train]
series_test  = series.loc[series.index > end_train]
exog_train = exog.loc[:end_train]
exog_test  = exog.loc[exog.index > end_train]

# Create forecaster
forecaster = ForecasterDirectMultiVariate(
    estimator          = LGBMRegressor(random_state=123, verbose=-1),
    level              = 'series_a',
    steps              = 10,
    lags               = [1, 2, 3],
    differentiation    = 1,
    dropna_from_series = Fa

## 4. Statistical

### 4.1 Arima (auto, daily → m=7, no exog)

In [13]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 365,
    target         = "y",
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "statistical",
    forecaster        = "ForecasterStats",
    estimator         = None,
    steps             = 30,
    frequency         = "D",
    forecaster_kwargs = {"stats_model": "Arima"},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Auto-ARIMA daily.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.stats import Arima
from skforecast.recursive import ForecasterStats

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster (Auto-ARIMA)
forecaster = ForecasterStats(
    estimator = Arima(order=None, seasonal=True, m=7),
)

# Fit
forecaster.fit(y=data_train['y'])

# Predict
steps = 30
predictions = forecaster.predict(steps=steps)
print(predictions)

# Evaluate on test set
actual = dat

### 4.2 Sarimax (monthly, m=12, with exog, native intervals)

In [14]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 365,
    target         = "y",
    index_type     = "datetime",
    frequency      = "MS",
    exog_columns   = ["temperature", "holiday"],
    end_train      = "2023-05-01",
)

plan = ForecastPlan(
    task_type         = "statistical",
    forecaster        = "ForecasterStats",
    estimator         = None,
    steps             = 12,
    frequency         = "MS",
    forecaster_kwargs = {"stats_model": "Sarimax"},
    interval          = [10, 90],
    interval_method   = "native",
    use_exog          = True,
    explanation       = "Sarimax with exog.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.stats import Sarimax
from skforecast.recursive import ForecasterStats

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('MS')

# Train/test split
end_train = '2023-05-01'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]
exog_features = ['temperature', 'holiday']

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster (Sarimax)
forecaster = ForecasterStats(
    estimator = Sarimax(order=None, seasonal_order=None, m=12),
)

# Fit
forecaster.fit(y=data_train['y'], exog=data_train[exog_features])

# Predict intervals (native)
steps = 1

### 4.3 Ets (monthly, m=12)

In [15]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 120,
    target         = "y",
    index_type     = "datetime",
    frequency      = "MS",
    end_train      = "2015-01-01",
)

plan = ForecastPlan(
    task_type         = "statistical",
    forecaster        = "ForecasterStats",
    estimator         = None,
    steps             = 12,
    frequency         = "MS",
    forecaster_kwargs = {"stats_model": "Ets"},
    interval_method   = None,
    use_exog          = False,
    explanation       = "ETS monthly.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.stats import Ets
from skforecast.recursive import ForecasterStats

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('MS')

# Train/test split
end_train = '2015-01-01'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster (ETS)
forecaster = ForecasterStats(
    estimator = Ets(m=12, model='ZZZ'),
)

# Fit
forecaster.fit(y=data_train['y'])

# Predict
steps = 12
predictions = forecaster.predict(steps=steps)
print(predictions)

# Evaluate on test set
actual = data_test['y'].iloc[:steps

### 4.4 Arar (no seasonal period)

In [16]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 200,
    target         = "y",
    index_type     = "datetime",
    frequency      = "15min",
    end_train      = "2024-01-02 04:00:00",
)

plan = ForecastPlan(
    task_type         = "statistical",
    forecaster        = "ForecasterStats",
    estimator         = None,
    steps             = 96,
    frequency         = "15min",
    forecaster_kwargs = {"stats_model": "Arar"},
    interval_method   = None,
    use_exog          = False,
    explanation       = "ARAR model.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.stats import Arar
from skforecast.recursive import ForecasterStats

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('15min')

# Train/test split
end_train = '2024-01-02 04:00:00'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster (ARAR)
forecaster = ForecasterStats(
    estimator = Arar(),
)

# Fit
forecaster.fit(y=data_train['y'])

# Predict
steps = 96
predictions = forecaster.predict(steps=steps)
print(predictions)

# Evaluate on test set
actual = data_test['y'].iloc[:steps]


## 5. Foundation

### 5.1 Chronos minimal (single series, no exog, no intervals)

In [17]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 365,
    target         = "y",
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "foundation",
    forecaster        = "ForecasterFoundation",
    estimator         = None,
    steps             = 30,
    frequency         = "D",
    forecaster_kwargs = {},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Chronos minimal.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.foundation import FoundationModel, ForecasterFoundation

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

series = data['y']

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
series_train = series.loc[:end_train]
series_test  = series.loc[series.index > end_train]

# Create foundation model (chronos-2-small)
model = FoundationModel(
    model_id       = 'autogluon/chronos-2-small',
    context_length = 2048,
    device_map     = 'auto',
)

# Create forecaster
forecaster = ForecasterFoundation(estimator=model)

# Fit (stores context only — no training)
forecaster.fit(series=series_train)

# Predict
steps = 30
predictions = forecaster.predict(steps=steps)
print(predictions)

# Evaluate on test set
actual = series_test.iloc[:steps]
pred 

### 5.2 Chronos custom (model_id, context_length, exog, custom quantiles)

In [18]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 500,
    target         = "y",
    index_type     = "datetime",
    frequency      = "h",
    exog_columns   = ["temperature"],
    end_train      = "2024-01-17",
)

plan = ForecastPlan(
    task_type         = "foundation",
    forecaster        = "ForecasterFoundation",
    estimator         = None,
    steps             = 24,
    frequency         = "h",
    forecaster_kwargs = {"model_id": "autogluon/chronos-2-base", "context_length": 4096},
    interval          = [20, 80],
    interval_method   = "native",
    use_exog          = True,
    explanation       = "Custom Chronos with exog.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.foundation import FoundationModel, ForecasterFoundation

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('h')

series = data['y']
exog = data[['temperature']]

# Train/test split
end_train = '2024-01-17'  # 80% of data, adjust to change the split point
series_train = series.loc[:end_train]
series_test  = series.loc[series.index > end_train]
exog_train = exog.loc[:end_train]
exog_test  = exog.loc[exog.index > end_train]

# Create foundation model (chronos-2-base)
model = FoundationModel(
    model_id       = 'autogluon/chronos-2-base',
    context_length = 4096,
    device_map     = 'auto',
)

# Create forecaster
forecaster = ForecasterFoundation(estimator=model)

# Fit (stores context only — no training)
forecaster.fit(series=series_train, exog=exog_train)

# Predict quantiles (nati

### 5.3 Foundation multi-series (wide format, levels in predict)

In [19]:
profile = DataProfile(
    data_format    = "wide",
    n_series       = 3,
    n_observations = 365,
    target         = ["series_a", "series_b", "series_c"],
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "foundation",
    forecaster        = "ForecasterFoundation",
    estimator         = None,
    steps             = 30,
    frequency         = "D",
    forecaster_kwargs = {},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Foundation multi-series.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.foundation import FoundationModel, ForecasterFoundation

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

series = data[['series_a', 'series_b', 'series_c']]

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
series_train = series.loc[:end_train]
series_test  = series.loc[series.index > end_train]

# Create foundation model (chronos-2-small)
model = FoundationModel(
    model_id       = 'autogluon/chronos-2-small',
    context_length = 2048,
    device_map     = 'auto',
)

# Create forecaster
forecaster = ForecasterFoundation(estimator=model)

# Fit (stores context only — no training)
forecaster.fit(series=series_train)

# Predict
steps = 30
predictions = forecaster.predict(steps=steps, levels=['series_a', 'series_b', 'series_c'])
prin

### 5.4 Foundation with default quantiles (no custom interval)

In [20]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 365,
    target         = "y",
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "foundation",
    forecaster        = "ForecasterFoundation",
    estimator         = None,
    steps             = 30,
    frequency         = "D",
    forecaster_kwargs = {},
    interval          = [10, 90],
    interval_method   = "native",
    use_exog          = False,
    explanation       = "Foundation default quantiles.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.foundation import FoundationModel, ForecasterFoundation

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

series = data['y']

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
series_train = series.loc[:end_train]
series_test  = series.loc[series.index > end_train]

# Create foundation model (chronos-2-small)
model = FoundationModel(
    model_id       = 'autogluon/chronos-2-small',
    context_length = 2048,
    device_map     = 'auto',
)

# Create forecaster
forecaster = ForecasterFoundation(estimator=model)

# Fit (stores context only — no training)
forecaster.fit(series=series_train)

# Predict quantiles (native)
steps = 30
predictions = forecaster.predict_quantiles(
    steps     = steps,
    quantiles = [0.1, 0.5, 0.9],
)
print(p